# FIT Data Ingestion & Validation Pipeline

This notebook implements a one-by-one ingestion workflow for activity files (strength, running, cycling, etc.):

1. **Load & inspect** existing raw files
2. **Define unified schema** for all activity types
3. **Validate** file structure and required columns
4. **Process individually** to avoid mixing invalid data
5. **Merge** validated datasets and export to staging/processed
6. **Log results** for audit trail

Start with the Strength FIT file already in `data/raw/`. Then add Running files one at a time.

## Section 1: Set Up Paths and Config

In [ ]:
import os
import sys
import logging
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent))

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configure paths
BASE_DIR = Path.cwd().parent  # theEagle/
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
STAGING_DIR = BASE_DIR / "data" / "staging"
STAGING_DIR.mkdir(exist_ok=True)

# Configuration
CONFIG = {
    "accepted_formats": [".fit"],
    "activity_types": ["running", "strength", "cycling", "swimming", "other"],
    "timestamp_format": "%Y-%m-%d %H:%M:%S",
}

print("Configuration:")
print(f"  Base directory: {BASE_DIR}")
print(f"  Raw files: {RAW_DIR}")
print(f"  Processed: {PROCESSED_DIR}")
print(f"  Staging: {STAGING_DIR}")
print(f"  Accepted formats: {CONFIG['accepted_formats']}")

## Section 2: Load Existing Strength FIT File

In [ ]:
# List all raw files
raw_files = sorted(RAW_DIR.glob("*.fit"))
print(f"Found {len(raw_files)} .fit file(s) in {RAW_DIR}:\n")

for fpath in raw_files:
    print(f"  {fpath.name}")

# Import FitParser
from src.fit_parser import FitParser

# Parse and inspect the strength file
if raw_files:
    fit_path = raw_files[0]
    print(f"\n{'='*60}")
    print(f"Parsing: {fit_path.name}")
    print(f"{'='*60}")
    
    parser = FitParser(fit_path)
    dfs = parser.parse()
    
    print(f"\nActivity type: {parser.activity_type}")
    print(f"Messages extracted: {len(dfs)}")
    print("\nMessage summary:")
    for msg_type, df in dfs.items():
        print(f"  {msg_type:<25} shape={df.shape}")
